# Digital Payment Fraud Pattern Analysis Using Spark
1. Set up Spark and upload the dataset
2. Load and preprocess the data
3. Analyse it with DataFrames, Spark SQL and RDDs
4. Explore it visually (EDA)
5. Identify suspicious transaction patterns
6. Engineer features, train and evaluate a model
7. (Optional) Compare with a Spark MLlib model
8. Export the model, configuration and metrics


# Setup

In [ ]:
!pip -q install pyspark scikit-learn joblib matplotlib pandas numpy


In [ ]:
# Libraries were installed in the previous cell.


In [ ]:
import json, os, shutil, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:          # running somewhere else (local Jupyter)
    IN_COLAB = False

spark = (SparkSession.builder
         .appName("DigitalPaymentFraudAnalysis")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
print("Spark version:", spark.version)

# Colours used in all charts
BLUE, NAVY, RED = "#2563EB", "#0B1F44", "#EF4444"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold"})

# 2. Upload and load the dataset

Expected columns: `trans_id, sender_id, receiver_id, payment_type, amount, oldbalance_sender, newbalance_sender, trans_hour, is_fraud`.

In [ ]:
if IN_COLAB:
    uploaded = files.upload()                 # pick digital_payment.csv
    DATA_PATH = list(uploaded.keys())[0]
else:
    DATA_PATH = "digital_payment.csv"          # change this if you run locally
print("Using file:", DATA_PATH)

raw = spark.read.csv(DATA_PATH, header=True, inferSchema=True)
raw.printSchema()
raw.show(5)
print("Rows:", raw.count())

# 3. Preprocessing
We check the columns, remove duplicates and missing values, standardise text, and drop impossible values.

In [ ]:
REQUIRED = ["trans_id", "sender_id", "receiver_id", "payment_type", "amount", "oldbalance_sender", "newbalance_sender", "trans_hour", "is_fraud"]
missing_cols = [c for c in REQUIRED if c not in raw.columns]
if missing_cols:
    raise ValueError(f"Training CSV missing required columns: {missing_cols}")

# Cast before testing missing values: failed numeric casts otherwise survive dropna.
clean = raw.dropDuplicates(["trans_id"])
for name in ["amount", "oldbalance_sender", "newbalance_sender"]:
    clean = clean.withColumn(name, F.col(name).cast("double"))
for name in ["trans_hour", "is_fraud"]:
    clean = clean.withColumn(name, F.col(name).cast("double"))
clean = (clean.withColumn("payment_type", F.upper(F.trim(F.col("payment_type").cast("string"))))
    .dropna(subset=REQUIRED)
    .filter(F.col("payment_type") != "")
    .filter(F.col("amount") > 0)
    .filter(F.col("trans_hour").between(0, 23) & (F.col("trans_hour") == F.floor("trans_hour")))
    .filter(F.col("is_fraud").isin(0.0, 1.0))
    .withColumn("trans_hour", F.col("trans_hour").cast("int"))
    .withColumn("is_fraud", F.col("is_fraud").cast("int")))
clean.cache()
valid_rows = clean.count()
print(f"Rows before: {raw.count()} | after cleaning: {valid_rows}")
if valid_rows == 0:
    raise ValueError("No valid rows remain after cleaning. Check CSV types, amounts, hours and binary labels.")


# 4. Analysis with Spark DataFrames

In [ ]:
total = clean.count()
fraud_total = clean.filter("is_fraud = 1").count()
print(f"Total: {total} | Fraud: {fraud_total} | Legitimate: {total - fraud_total} | Fraud %: {100*fraud_total/total:.2f}")

by_type = (clean.groupBy("payment_type")
    .agg(F.count("*").alias("transactions"),
         F.sum("is_fraud").alias("fraud_cases"),
         F.round(F.avg("amount"), 2).alias("avg_amount"),
         F.round(100 * F.avg("is_fraud"), 2).alias("fraud_rate_pct"))
    .orderBy(F.desc("fraud_rate_pct")))
by_type.show()

# 5. Analysis with Spark SQL

In [ ]:
clean.createOrReplaceTempView("transactions")

print("Fraud rate by hour of day")
spark.sql('''
    SELECT trans_hour, COUNT(*) AS transactions, SUM(is_fraud) AS fraud_cases,
           ROUND(100 * AVG(is_fraud), 2) AS fraud_rate_pct
    FROM transactions GROUP BY trans_hour ORDER BY trans_hour
''').show(24)

print("Fraud rate by amount band")
spark.sql('''
    SELECT CASE WHEN amount < 100 THEN '1) < 100'
                WHEN amount < 500 THEN '2) 100 - 499'
                WHEN amount < 2000 THEN '3) 500 - 1,999'
                WHEN amount < 5000 THEN '4) 2,000 - 4,999'
                ELSE '5) 5,000+' END AS amount_band,
           COUNT(*) AS transactions, SUM(is_fraud) AS fraud_cases,
           ROUND(100 * AVG(is_fraud), 2) AS fraud_rate_pct
    FROM transactions GROUP BY 1 ORDER BY 1
''').show()

# 6. Analysis with RDDs
RDDs are Spark's low-level API. Here we count transactions per sender and fraud cases per receiver with `map` / `reduceByKey`.

In [ ]:
rdd = clean.select("sender_id", "receiver_id", "is_fraud").rdd

sender_counts = rdd.map(lambda r: (r["sender_id"], 1)).reduceByKey(lambda a, b: a + b)
print("Top 10 most active senders:")
for sender, n in sender_counts.takeOrdered(10, key=lambda x: -x[1]):
    print(f"  {sender}: {n} transactions")

fraud_per_receiver = (rdd.filter(lambda r: r["is_fraud"] == 1)
                         .map(lambda r: (r["receiver_id"], 1))
                         .reduceByKey(lambda a, b: a + b))
print("\nTop 10 receivers by fraud cases:")
for receiver, n in fraud_per_receiver.takeOrdered(10, key=lambda x: -x[1]):
    print(f"  {receiver}: {n} fraud cases")

# 7. Exploratory data analysis (charts)

In [ ]:
eda = clean.toPandas()
legit, fraud = eda[eda.is_fraud == 0], eda[eda.is_fraud == 1]

fig, ax = plt.subplots(2, 2, figsize=(14, 9))

counts = eda.payment_type.value_counts()
ax[0, 0].bar(counts.index, counts.values, color=BLUE)
ax[0, 0].set_title("Transactions by payment type"); ax[0, 0].set_ylabel("Transactions")

rate = eda.groupby("payment_type").is_fraud.mean().mul(100).sort_values(ascending=False)
ax[0, 1].bar(rate.index, rate.values, color=NAVY)
ax[0, 1].set_title("Fraud rate by payment type (%)")

bins = np.logspace(np.log10(eda.amount.min()), np.log10(eda.amount.max()), 40)
ax[1, 0].hist(legit.amount, bins=bins, alpha=.7, color=BLUE, label="Legitimate")
ax[1, 0].hist(fraud.amount, bins=bins, alpha=.8, color=RED, label="Fraud")
ax[1, 0].set_xscale("log"); ax[1, 0].set_title("Amount distribution (log scale)"); ax[1, 0].legend()

hourly = eda.groupby("trans_hour").is_fraud.sum()
ax[1, 1].bar(hourly.index, hourly.values, color=BLUE)
ax[1, 1].set_title("Fraud cases by hour of day"); ax[1, 1].set_xlabel("Hour")

plt.tight_layout(); plt.show()

# 8. Suspicious transaction patterns

In [ ]:
# Simple fraud pattern analysis using familiar DataFrame functions
# 1. Find unusually large payments
p95_amount = clean.approxQuantile("amount", [0.95], 0.01)[0]
print("High amount cutoff:", round(p95_amount, 2))
clean.filter(F.col("amount") >= p95_amount).select("trans_id", "amount", "payment_type").show(10)

# 2. Find senders who made several payments in the same hour
sender_hour = clean.groupBy("sender_id", "trans_hour").count()
busy_senders = sender_hour.filter(F.col("count") >= 3)
print("Frequent sender-hour combinations:", busy_senders.count())
busy_senders.show(10)

# 3. Repeated sender, receiver and amount combinations
duplicates = clean.groupBy("sender_id", "receiver_id", "amount").count().filter("count >= 2")
print("Repeated payment combinations:", duplicates.count())
duplicates.show(10)


|# 8b. Rule-based risk scoring (points system)

In [ ]:
# Understand risk rules with simple Spark filters and grouping.
# These point values also appear in risk_rules.json for Streamlit.
POINTS = {"high_amount": 20, "frequent_sender": 25, "duplicate": 20,
          "failed_attempts": 15, "unusual_location": 20, "unusual_hour": 10}

# Make a simple list of suspicious transaction IDs for each rule.
high_ids = clean.filter(F.col("amount") >= p95_amount).select("trans_id").distinct()
busy_ids = clean.join(busy_senders.select("sender_id", "trans_hour"),
                      ["sender_id", "trans_hour"], "inner").select("trans_id").distinct()
duplicate_ids = clean.join(duplicates.select("sender_id", "receiver_id", "amount"),
                           ["sender_id", "receiver_id", "amount"], "inner").select("trans_id").distinct()

scored = clean.join(high_ids.withColumn("high_flag", F.lit(1)), "trans_id", "left")
scored = scored.join(busy_ids.withColumn("frequent_flag", F.lit(1)), "trans_id", "left")
scored = scored.join(duplicate_ids.withColumn("duplicate_flag", F.lit(1)), "trans_id", "left")

for name in ["high_flag", "frequent_flag", "duplicate_flag"]:
    scored = scored.fillna(0, subset=[name])

scored = scored.withColumn("rule_score",
    F.col("high_flag") * POINTS["high_amount"] +
    F.col("frequent_flag") * POINTS["frequent_sender"] +
    F.col("duplicate_flag") * POINTS["duplicate"])

# Optional columns are used only when available.
if "failed_attempts" in clean.columns:
    scored = scored.withColumn("rule_score", F.col("rule_score") +
        F.when(F.col("failed_attempts").cast("int") >= 3, POINTS["failed_attempts"]).otherwise(0))
if "location" in clean.columns:
    location_counts = clean.groupBy("location").count()
    rare_locations = location_counts.filter(F.col("count") < clean.count() * 0.05).select("location")
    scored = scored.join(rare_locations.withColumn("rare_flag", F.lit(1)), "location", "left")
    scored = scored.fillna(0, subset=["rare_flag"])
    scored = scored.withColumn("rule_score", F.col("rule_score") + F.col("rare_flag") * POINTS["unusual_location"])
scored = scored.withColumn("rule_score", F.col("rule_score") +
    F.when(F.col("trans_hour").isin([0,1,2,3,4,5]), POINTS["unusual_hour"]).otherwise(0))
scored = scored.withColumn("rule_score", F.least(F.col("rule_score"), F.lit(100)))
scored = scored.withColumn("rule_flag", F.col("rule_score") >= 20)
scored.select("trans_id", "rule_score", "rule_flag", "is_fraud").show(15)
print("Rule-flagged transactions:", scored.filter("rule_flag = true").count())
print("Remember: a rule flag is NOT confirmed fraud.")


# 9. Feature engineering (in Spark)

In [ ]:
NUMERIC_FEATURES = ["amount", "oldbalance_sender", "newbalance_sender", "trans_hour",
                    "balance_diff", "amount_to_balance", "is_night", "zero_balance_after"]
CATEGORICAL_FEATURES = ["payment_type"]
TARGET = "is_fraud"

features_sdf = (clean
    .withColumn("balance_diff", F.round(F.col("oldbalance_sender") - F.col("newbalance_sender") - F.col("amount"), 2))
    .withColumn("amount_to_balance", F.round(F.col("amount") / (F.col("oldbalance_sender") + 1), 4))
    .withColumn("is_night", F.when((F.col("trans_hour") <= 5) | (F.col("trans_hour") >= 23), 1).otherwise(0))
    .withColumn("zero_balance_after", F.when(F.col("newbalance_sender") == 0, 1).otherwise(0)))

# The dataset is small, so it is safe to bring it to pandas for scikit-learn
pdf = features_sdf.select("trans_id", *NUMERIC_FEATURES, *CATEGORICAL_FEATURES, TARGET).toPandas()
print(pdf.shape)
pdf.head()

# 10. Train and evaluate the model

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

RANDOM_STATE, TEST_SIZE = 42, 0.2
X = pdf[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = pdf[TARGET].astype(int)
counts = y.value_counts()
if set(counts.index) != {0, 1} or counts.min() < 2:
    raise ValueError(f"Training requires both label classes with at least 2 samples each; got {counts.to_dict()}.")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
train_counts = y_train.value_counts()
if set(train_counts.index) != {0, 1} or train_counts.min() < 2:
    raise ValueError(f"Insufficient minority-class examples after split: {train_counts.to_dict()}")
print("Train:", X_train.shape, "| Test:", X_test.shape, "| Fraud in test:", int(y_test.sum()))

def make_pipeline(estimator):
    prep = ColumnTransformer([("payment_type", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)], remainder="passthrough")
    return Pipeline([("prep", prep), ("model", estimator)])

candidates = {
    "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}
cv_splits = min(5, int(train_counts.min()))
cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
cv_f1 = {}
for name, est in candidates.items():
    cv_f1[name] = float(cross_val_score(make_pipeline(est), X_train, y_train, cv=cv, scoring="f1").mean())
    print(f"{name:18s} CV F1 = {cv_f1[name]:.3f}")
best_name = max(cv_f1, key=cv_f1.get)
print("Selected model:", best_name)
model = make_pipeline(candidates[best_name]).fit(X_train, y_train)


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
THRESHOLD = 0.5
proba = model.predict_proba(X_test)[:, list(model.classes_).index(1)]
pred = (proba >= THRESHOLD).astype(int)
has_both_test_classes = y_test.nunique() == 2
metrics = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "precision": float(precision_score(y_test, pred, zero_division=0)) if has_both_test_classes else None,
    "recall": float(recall_score(y_test, pred, zero_division=0)) if has_both_test_classes else None,
    "f1": float(f1_score(y_test, pred, zero_division=0)) if has_both_test_classes else None,
    "roc_auc": float(roc_auc_score(y_test, proba)) if has_both_test_classes else None,
}
if not has_both_test_classes:
    print("WARNING: Single-class test set; precision, recall, F1 and ROC-AUC are omitted.")
cm = confusion_matrix(y_test, pred, labels=[0, 1])
for k, v in metrics.items():
    print(f"{k:10s}: {v:.4f}" if v is not None else f"{k:10s}: N/A")
print("\n", classification_report(y_test, pred, labels=[0, 1], target_names=["Legitimate", "Fraud"], zero_division=0))
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["Legitimate", "Fraud"])
ax[0].set_yticks([0, 1]); ax[0].set_yticklabels(["Legitimate", "Fraud"])
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("Actual"); ax[0].set_title("Confusion matrix (test set)")
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=16,
                   color="white" if cm[i, j] > cm.max() / 2 else NAVY)
feature_names = [n.split("__", 1)[-1] for n in model.named_steps["prep"].get_feature_names_out()]
importances = model.named_steps["model"].feature_importances_
order = np.argsort(importances)
ax[1].barh(np.array(feature_names)[order], importances[order], color=BLUE)
ax[1].set_title("Feature importance")
plt.tight_layout(); plt.show()
feature_importances = {n: float(v) for n, v in zip(feature_names, importances)}


# 11. Optional extension: Spark MLlib

Our project uses **basic PySpark** for data loading, filtering, grouping, Spark SQL and RDD analysis.
We use **scikit-learn** to train the main machine-learning fraud classifier.
Spark MLlib would be an optional future comparison, but is not required for the Streamlit app.


The optional advanced Spark MLlib experiment has been removed to keep the notebook suitable for the classroom presentation. The existing scikit-learn fraud model above is still trained, evaluated and exported for the Streamlit website.

# 12. Export for the Streamlit app
Training summaries are model metadata; the Streamlit dashboard always computes totals from its currently selected CSV.


In [ ]:
EXPORT_DIR = "export"
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
os.makedirs(f"{EXPORT_DIR}/models"); os.makedirs(f"{EXPORT_DIR}/data")

# 1) Model
joblib.dump(model, f"{EXPORT_DIR}/models/fraud_model.joblib")

# 2) Feature configuration
feature_config = {
    "target": TARGET,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "optional_rule_columns": ["sender_id", "receiver_id", "location", "failed_attempts", "timestamp"],
    "required_input_columns": ["payment_type", "amount", "oldbalance_sender", "newbalance_sender", "trans_hour"],
    "payment_types_seen": sorted(pdf["payment_type"].unique().tolist()),
    "feature_formulas": {
        "balance_diff": "round(oldbalance_sender - newbalance_sender - amount, 2)",
        "amount_to_balance": "round(amount / (oldbalance_sender + 1), 4)",
        "is_night": "1 if trans_hour <= 5 or trans_hour >= 23 else 0",
        "zero_balance_after": "1 if newbalance_sender == 0 else 0",
    },
    "default_threshold": THRESHOLD,
    "sklearn_version": sklearn.__version__,
}

# 3) Metrics + dataset summary (all computed from the data above)
fraud_mask = pdf[TARGET] == 1
dataset_summary = {
    "total_transactions": int(len(pdf)),
    "fraud_cases": int(fraud_mask.sum()),
    "legitimate_transactions": int((~fraud_mask).sum()),
    "fraud_percentage": float(100 * fraud_mask.mean()),
    "total_amount": float(pdf["amount"].sum()),
    "average_amount": float(pdf["amount"].mean()),
    "fraud_amount": float(pdf.loc[fraud_mask, "amount"].sum()),
    "by_payment_type": [{"payment_type": k, "transactions": int(len(g)), "fraud_cases": int(g[TARGET].sum())}
                        for k, g in pdf.groupby("payment_type")],
    "by_hour": [{"hour": int(k), "transactions": int(len(g)), "fraud_cases": int(g[TARGET].sum())}
                for k, g in pdf.groupby("trans_hour")],
}
model_metrics = {
    "model_name": best_name,
    "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "sklearn_version": sklearn.__version__,
    "random_state": RANDOM_STATE, "test_size": TEST_SIZE,
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "threshold": THRESHOLD,
    **metrics,
    "confusion_matrix": cm.tolist(),          # [[TN, FP], [FN, TP]]
    "cv_f1": cv_f1,
    "feature_importances": feature_importances,
    "dataset_summary": dataset_summary,
}
if "spark_metrics" in globals():
    model_metrics["spark_mllib_random_forest"] = {k: float(v) for k, v in spark_metrics.items()}

with open(f"{EXPORT_DIR}/data/feature_columns.json", "w") as f:
    json.dump(feature_config, f, indent=2)
with open(f"{EXPORT_DIR}/data/model_metrics.json", "w") as f:
    json.dump(model_metrics, f, indent=2)

risk_rules = {
    "description": "Heuristic risk points, NOT fraud probabilities or verified labels.",
    "weights": {"Unusually high amount": 20, "Many transactions in a short period": 25,
                "Repeated failed attempts": 15, "Unusual location": 20, "Duplicate payment pattern": 20,
                "Unusual transaction hour": 10},
    "high_amount_quantile": 0.95,
    "frequency_count": 3,
    "failed_attempts_threshold": 3,
    "rare_location_share": 0.05,
    "unusual_hours": [0, 1, 2, 3, 4, 5],
    "flag_threshold": 20,
    "risk_categories": {"Medium": 20, "High": 40, "Critical": 70}
}
with open(f"{EXPORT_DIR}/data/risk_rules.json", "w") as f:
    json.dump(risk_rules, f, indent=2)
shutil.make_archive("fraud_model_artifacts", "zip", EXPORT_DIR)
print("Created fraud_model_artifacts.zip containing:")
for root, _, names in os.walk(EXPORT_DIR):
    for n in names:
        print("  ", os.path.relpath(os.path.join(root, n), EXPORT_DIR))

In [ ]:
# Download the ZIP to your computer (then follow the README to place the files)
if IN_COLAB:
    files.download("fraud_model_artifacts.zip")